In [1]:
#导入包
import json
import Utils
import importlib
import jieba
import networkx as nx
import pandas as pd
import KnowledgeFeatureExtraction as kfe
import KnowledgeNetwork as kn
from tqdm import tqdm
importlib.reload(Utils)
importlib.reload(kfe)
importlib.reload(kn)

<module 'KnowledgeNetwork' from 'F:\\PythonProjects\\nlu4mwps\\KnowledgeNetwork.py'>

In [2]:
raw_df=Utils.read_dataset("data/questions_all.json")
math_concepts_freq=Utils.read_text("data/math_concepts.txt")
math_concepts={text.split(",")[1] for text in math_concepts_freq}
math_concepts.update(set(["量","比","数","分式","归一","四则","棱长","估算","性质","计量","正方体","应用题","一元一次","二元一次","一元二次"]))
for mc in math_concepts:
    jieba.add_word(mc)
jieba.del_word("应用")

def seg4MathConcepts(text):
    text=text.strip()
    cut_list={word for word in jieba.cut(text,cut_all=True) if word in math_concepts}
    if len(cut_list)==0:
        return text
    return " ".join(cut_list)

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\jimso\AppData\Local\Temp\jieba.cache
Loading model cost 0.723 seconds.
Prefix dict has been built successfully.


In [3]:
raw_df.head(2)

,id,type,original_text,equation,ans,examination_point,analyse,raw_text,pos,num_loc,sni_loc,constant,unk,ops
0,371,0,某段 高速公路 的 路基 长 120 千 米 ， 宽 50 米 。 这 段 高速公路 占...,x = 120.0 * 1000.0 * 50.0 / 10000.0,[600],长方形、正方形的面积,根据长方形的面积公式:S＝a*b，把数据代入公式解答即可,某段高速公路的路基长120千米，宽50米。这段高速公路占地多少公顷？,r n uj nrt ns m q x a m m x r q n v m q x,"[5, 10]","[5, 10]","[1000.0, 10000.0]",[x],"[=, *, /]"
1,15226,1,甲 、 乙 2 车 同时 从 A 城 去 B 城 ， 甲车 每 小时 行 35 千 米 ...,35.0 * x = 40.0 * ( x - 0.5 ),[4],一元一次方程的应用,相等关系：甲车行驶的时间-乙车行驶的时间=0.5。可设路程表示时间，列方程求解。,甲、乙两车同时从A城去B城，甲车每小时行35千米，乙车每小时行40千米，结果乙比甲提前0.5...,n x n m n c p eng n v eng n x n r n n m q x n ...,"[3, 17, 25, 32]","[17, 25, 32]",[],[x],"[(, ), -, *, =]"


In [4]:
raw_df["knowledge_point"]=raw_df['examination_point'].apply(seg4MathConcepts)

In [7]:
raw_df.to_excel("mwp_all_kps.xlsx")

In [28]:
kfe_extractor=kfe.KnowledgeFeatureExtractor()
all_knowledge_lists=[knowledege_points.split(" ") for knowledege_points in raw_df["knowledge_point"]]
kfe_extractor.build_cooccurrence_network(all_knowledge_lists)

In [29]:
raw_df.head()

,id,type,original_text,equation,ans,examination_point,analyse,raw_text,pos,num_loc,sni_loc,constant,unk,ops,knowledge_point
0,371,0,某段 高速公路 的 路基 长 120 千 米 ， 宽 50 米 。 这 段 高速公路 占...,x = 120.0 * 1000.0 * 50.0 / 10000.0,[600],长方形、正方形的面积,根据长方形的面积公式:S＝a*b，把数据代入公式解答即可,某段高速公路的路基长120千米，宽50米。这段高速公路占地多少公顷？,r n uj nrt ns m q x a m m x r q n v m q x,"[5, 10]","[5, 10]","[1000.0, 10000.0]",[x],"[=, *, /]",正方形 长方形 面积
1,15226,1,甲 、 乙 2 车 同时 从 A 城 去 B 城 ， 甲车 每 小时 行 35 千 米 ...,35.0 * x = 40.0 * ( x - 0.5 ),[4],一元一次方程的应用,相等关系：甲车行驶的时间-乙车行驶的时间=0.5。可设路程表示时间，列方程求解。,甲、乙两车同时从A城去B城，甲车每小时行35千米，乙车每小时行40千米，结果乙比甲提前0.5...,n x n m n c p eng n v eng n x n r n n m q x n ...,"[3, 17, 25, 32]","[17, 25, 32]",[],[x],"[(, ), -, *, =]",应用 一元一次 方程 一元一次方程
2,13789,1,有 1 块 面积 为 150 亩 的 绿化 工程 面向 全 社会 公开招标 。 现有 甲...,150.0 / ( 1.0 / 2.0 * x ) - 150.0 / x = 10.0,[15],分式方程的应用,求的是时间，工作总量为150，一定是根据工作效率来列等量关系，本题的关键描述语是：甲队比乙队...,有一块面积为150亩的绿化工程面向全社会公开招标。现有甲、乙两工程队前来竞标，甲队计划比规定...,v m q n p m m uj n n n a n n x b n x n m n t n...,"[1, 5, 19, 30, 32, 46]","[1, 5, 19, 30, 32, 46]",[],[x],"[(, ), -, *, =, /]",方程 应用
3,11955,1,领队 小 李 带 驴 友团 去 某 景区 ， 1 共 12 人 。 景区 门票 成人 每...,60.0 * x + 60.0 * 0.5 * ( 12.0 - x ) = 600.0,[8],一元一次方程的应用——方案选择,设驴友团中有x名成人1则有(12-x)名未成年人，根据购票总价=60*成人人数+60*0.5...,领队小李带驴友团去某景区，一共12人。景区门票成人每张60元，未成年人按成人票价的五折优惠：...,n a nr v n n v r n x m n m n x n n n r m m x l...,"[10, 12, 19, 27, 36]","[12, 19, 27, 36]",[],[x],"[(, ), -, +, *, =]",方程 一元一次 应用 一元一次方程 选择
4,485,0,1 块 正方形 地 的 边长 是 300 米 ， 每 公顷 收 稻谷 8 吨 ， 那么...,x = 300.0 * 300.0 / 10000.0 * 8.0,[72],长方形、正方形的面积,根据正方形的面积＝边长*边长计算出面积，然后用面积乘单位面积的产量，即可解答问题,一块正方形地的边长是300米，每公顷收稻谷8吨，那么这块地收稻谷多少吨？,m q n uv uj n v m m x zg q v n m m x r r n n m...,"[0, 7, 14]","[7, 14]",[10000.0],[x],"[=, *, /]",正方形 长方形 面积


In [ ]:
feature_df=pd.DataFrame()
for i in tqdm(range(len(all_knowledge_lists)), desc="Preprocessing data:"):
    feature_dict=kfe_extractor.extract_all_features(knowledge_list=all_knowledge_lists[i])
    temp_df=pd.DataFrame(feature_dict,index=[0])
    feature_df = pd.concat([feature_df, temp_df], axis=0).reset_index(drop=True)
feature_df.head()

In [9]:
# kfe_extractor.visualize_knowledge_network(top_n=200)

In [30]:
kn_analyzer=kn.KnowledgeNetworkAnalyzer(knowledge_graph=kfe_extractor.knowledge_cooccurrence_network)

In [ ]:
kn_analyzer.analyze_hierarchy()

In [ ]:
kn_analyzer.visualize_hierarchical_structure()